In [ ]:
#| hide
from vishalakshi import *
from fastcore.all import *

# concepts

> what retrieval returns, which encoder wrote it, and why one file holds several vector spaces

The [README](index.html) shows what vishalakshi does. This page is the handful of decisions you make
when a default stops being right: a corpus that wants a different embedder, an answer that should
come from a bigger model, two shelves that should not be compared.

In [ ]:
from tempfile import mkdtemp
from vishalakshi import Vault

v = Vault(Path(mkdtemp())/'vault.db')
v.note('federate fuses the legs by rank because they share no vector space: the vault embeds '
       'prose, kosha embeds identifiers, ripgrep embeds nothing.', tags=['retrieval'])
v.add('# Late chunking\n\n## Method\n\nEmbed the whole document, then pool per chunk, so a chunk '
      'keeps the context around it.\n\n## Results\n\nEvaluated on BEIR.', 'Late chunking', kind='note')
v.connect()
v.stats()

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'docs': 2,
 'nodes': 6,
 'chunks': 3,
 'encoder': 'model2vec',
 'entities': 21,
 'path': '/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmph5ysy_7x/vault.db',
 'by_kind': {'note': 2}}

## What retrieval actually returns

`v.context(q)` is the primitive underneath `ask`. It returns whole sections with breadcrumbs and `node_id`s you can `read` back.


In [ ]:
c = v.context('why are rankings fused instead of distances?', sections=4, related=4)
for s in c.results: print(f'{len(s.text):5}  {s.breadcrumb}')

  141  federate fuses the legs by rank because they share no vector space: the vault em
   86  Late chunking › Method
   18  Late chunking › Results
  234  repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/code.py:57
  154  grep › index.ipynb:161
  600  repo › /Users/71293/code/personal/orgs/vishalakshi/vishalakshi/extract.py:179
   89  grep › 07_concepts.ipynb:101


In [ ]:
for s in c.related: print(f'{s.via:6}  {s.breadcrumb}')

`results` answer the question. `related` are associations along the entity graph, useful before you know what to search for, not wired into ranking.


In [ ]:
c.encoder

'minishlab/potion-multilingual-128M (256d, float16, model2vec)'

`c.encoder` always says which embedder answered. Hashing fallback costs +0.089 recall@10 on known-item queries (`evals/encoder.py`).


## Models and backends

Naming a model is naming it to rishi. LiteRT is the default local path; hosted ids need their keys. Details that change with rishi stay there; this page only shows what the vault passes through.


In [ ]:
from rishi.core import Chat, infer_runtime
from vishalakshi.ask import dflt_model

# rishi's backend modules name their models — `rishi.litert.gemma4_e2b` is this id — but importing
# one imports its backend, so a page about ids talks about them as ids
gemma4_e2b = 'litert-community/gemma-4-E2B-it-litert-lm'
dflt_model, gemma4_e2b     # $VISHALAKSHI_MODEL if set, else rishi's own small local default

In [ ]:
{m: infer_runtime(m) for m in (gemma4_e2b, 'mlx-community/Qwen3-4B-4bit', 'my-local.gguf',
                               'gpt-5.6-luna', 'gemma-3-4b-it-int4')}

In [ ]:
#| eval: false
v.ask('why are rankings fused?', model=gemma4_e2b)             # an id: rishi knows what runs it
v.ask('why are rankings fused?', model='llama/my-local.gguf')  # a prefix when the id cannot say
v.ask('why are rankings fused?', model='mlx-community/Qwen3-4B-4bit',
      chat_kw=dict(temp=0, think=True))          # the rest of rishi's constructor

In [ ]:
from fastcore.test import test_fail
# `model=` plus `chat_kw=` is the whole model API: anything rishi's constructor takes, and rishi's
# own error when the id says nothing about which backend should run it
test_fail(lambda: Chat('gemma-3-4b-it-int4'), contains='backend')

A reasoning model's `<think>` block goes to `r.thinking` rather than `r.answer`: it names sections
it then discards, and citations are read off the answer.

Retrieval never needs the network. Only answering with a hosted model does.

## Encoders

Default is a static multilingual encoder, 256d float16. `offline=True` hashes instead of embedding (+0.089 recall@10 on known-item queries, `evals/encoder.py`). A shelf records the encoder that wrote it.


In [ ]:
Vault(':memory:', offline=True).enc.note      # never attempt a download — also the CI default

'char-n-gram hashing (256d) — lexical only; pass encoder= or restore network access for real semantics'

In [ ]:
#| eval: false
Vault(encoder='minishlab/potion-science-32M')   # pick a different one

Vault('/Users/71293/.vishalakshi/vault.db': 0 docs, 0 chunks, 0 entities, encoder=model2vec)

Both encoders store float16, litesearch's default width. Mixing models inside one ANN index is not supported: one index is one vector space.


The vault takes any encoder [litesearch](https://github.com/vedicreader/litesearch) ships, by name:

In [ ]:
from vishalakshi.core import ENCODERS, enc_spec

{k: (m if isinstance(m, str) else m['model']) for k, m in ENCODERS.items()}

{'default': 'minishlab/potion-multilingual-128M',
 'multilingual': 'minishlab/potion-multilingual-128M',
 'retrieval': 'minishlab/potion-retrieval-32M',
 'science': 'minishlab/potion-science-32M',
 'code': 'minishlab/potion-code-16M-v2',
 'sanskritgemma': 'karthikrajgopal/sanskritgemma-256',
 'gemma': 'onnx-community/embeddinggemma-300m-ONNX',
 'bge-micro': 'TaylorAI/bge-micro-v2',
 'modernbert': 'nomic-ai/modernbert-embed-base',
 'nomic': 'nomic-ai/nomic-embed-text-v1.5'}

Static models are lookup tables (ms/doc, no GPU). Transformers cost more and win only some genres; the default static encoder is the measured pick.


In [ ]:
enc_spec('science'), enc_spec('bge-micro')[1], enc_spec('gemma')[0]['model']

(('minishlab/potion-science-32M', 'static'),
 'onnx',
 'onnx-community/embeddinggemma-300m-ONNX')

One ANN index is one vector space. Different encoders need different shelves; that is what `SHELVES` / `KIND_SHELF` are for.


In [ ]:
from vishalakshi.core import SHELVES, KIND_SHELF

SHELVES, KIND_SHELF

({'store': 'default',
  'papers': 'science',
  'sanskrit': 'gemma',
  'sanskrit-fast': 'default',
  'code': 'code',
  'data': 'retrieval'},
 {'arxiv': 'papers', 'sanskrit': 'sanskrit'})

In [ ]:
papers = v.shelf('papers')        # no encoder named: the registry knows it is a science model
papers.add('# Late chunking\n\nWe evaluate contextual chunk embeddings on BEIR.', 'a paper')

[(s['store'], s['encoder'], s['docs']) for s in v.shelves()]

[('store', 'minishlab/potion-multilingual-128M', 2),
 ('papers', 'minishlab/potion-science-32M', 1)]

`KIND_SHELF` routes acquisition into a shelf. `reshelf` moves a document after the fact; `elsewhere` searches neighbouring shelves when the answer may not be on the current one.


In [ ]:
v.route('arxiv').name, v.route('web').name

('papers', 'store')

A PDF is as likely an invoice as a paper. Only `categorize` can tell, and only once the document is filed; then `reshelf` moves it.


In [ ]:
v.add('# Attention is all you need\n\n## Abstract\n\nWe propose the Transformer.\n\n'
      '## Introduction\n\nRelated work [1] et al.\n\n## References\n\ndoi:10.1/x',
      'attention', source='/inbox/attention.pdf')

r = v.reshelf('/inbox/attention.pdf', llm='never')
r.doctype, r.was, r.store, r.moved

('paper', 'store', 'papers', True)

In [ ]:
v.doc('/inbox/attention.pdf'), v.shelf('papers').doc('/inbox/attention.pdf')['title']

(None, 'attention')

A move is a re-ingest: the other shelf has a different encoder, so vectors are remade. Meta that should survive goes in `doc_marks`.


`federate` fuses shelves by rank alongside kosha and ripgrep. `elsewhere` appends a couple of sections from each other shelf for `context` / `ask`.


In [ ]:
v.federate('contextual chunk embeddings', repo=False, grep=False).legs

{'prose': 3, 'shelf:papers': 4}

`elsewhere` tags where each neighbour came from. Read with `store=` when the `node_id` is not on the current shelf.


In [ ]:
e = v.elsewhere('contextual chunk embeddings')
L(e).map(lambda r: (r.store, r.breadcrumb))


[('papers', 'papers › a paper'),
 ('papers', 'papers › attention › Attention is all you need › Abstract')]

In [ ]:
v.read(e[0].node_id, store=e[0].store)['text'][:80]

'# Late chunking\n\nWe evaluate contextual chunk embeddings on BEIR.'

A shelf records the encoder that wrote it, reopens with the right one, and warns on mismatch. Rebuild the shelf if you change encoders on purpose.
